# 01 — Exploratory data analysis

**Objective:** understand 5-class damage severity, check raw schema and quantify missingness. Run from the repo root or notebook folder. No labels from the Kaggle test set are used.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if Path('scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scripts.preprocess import read_competition_data, parse_grades
train, test, sample = read_competition_data(ROOT/'data'/'structural-damage-classification.zip')
print('Train / Test / Sample:', train.shape, test.shape, sample.shape)
display(train.head(3)); display(train.dtypes.to_frame('dtype'))

In [ ]:
y = parse_grades(train['damage_grade'])
freq = pd.Series(y).value_counts().sort_index()
display(pd.DataFrame({'n':freq, 'share':freq / len(y)}))
fig, ax = plt.subplots(figsize=(7,4))
freq.plot.bar(ax=ax, title='Damage grade distribution (training only)')
ax.set(xlabel='Grade', ylabel='Buildings'); plt.show()

In [ ]:
missing = (train.isna().mean() * 100).sort_values(ascending=False)
display(missing.to_frame('missing_%').head(30))
cat_cols = train.select_dtypes(include=['object','string','category']).columns.drop('building_id', errors='ignore')
for c in cat_cols[:12]:
    print(c, train[c].value_counts(dropna=False).head(8).to_dict())
num = train.select_dtypes(include='number').columns.drop('damage_grade', errors='ignore')
display(train[num].describe().T if len(num) else pd.DataFrame())

## Interpretation guide

Record whether classes are imbalanced, which construction features are sparse, and whether survey fields represent **pre**-earthquake structure. This notebook reports descriptive patterns; it makes no causal claims. Inspect the separate held-out evaluation before replacing an existing leaderboard submission.